# اليوم الثاني — مختبر 3A: تصنيف النصوص
## Day 2 — Lab 3A: Text Classification

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** Core → Explore → Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/develop/notebooks/03_text_classification.ipynb)

> نبني Baseline أولًا، نثبت عدم تسرب المجموعات، ثم ننفذ خطوة ضبط فعلية لـDistilmBERT متعدد اللغات.
>
> Build a baseline, prove group isolation, then perform a real fine-tuning step on multilingual DistilBERT.

**علامة النجاح:** `DAY2_NOTEBOOK3_CORE=PASS`.

## قواعد المختبر

- البيانات اصطناعية ولا تحتوي حالات أو أشخاصًا حقيقيين.
- GPU في Colab Free غير مضمون؛ CPU fallback يجمد المشفر ويدرب task head.
- لا نرفع model weights أو cache إلى GitHub.
- نتائج هذه العينة الصغيرة تسمى `MEASURED_SMOKE`، وليست أداءً إنتاجيًا.
- نفذ **Runtime → Run all**. لا تتجاوز خلية فاشلة.

In [ ]:
# تثبيت نسخ اليوم الثاني فقط عند الحاجة
import importlib.metadata
import importlib.util
import subprocess
import sys

REQUIRED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "scikit-learn": "1.9.0",
}
needs_install = []
for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != expected:
        needs_install.append(f"{distribution}=={expected}")

if needs_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needs_install])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is required. Open this notebook in Google Colab.")

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

In [ ]:
import csv
import io
import json
import math
import os
import random
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1) تحميل البيانات وفحصها

يحاول الدفتر قراءة ملف الدورة من GitHub. إذا تعذر، يستخدم عينة اصطناعية مدمجة حتى لا يتوقف Core بسبب رابط البيانات. النموذج نفسه يحتاج تنزيلًا أول مرة.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/develop/data/sample/bayan_day2_classification.csv"
FALLBACK_ROWS = [{"example_id":"F-001","group_id":"DG-A","split":"train","language":"ar","text":"تعذر تسجيل الدخول إلى البوابة","topic":"digital_service"},{"example_id":"F-002","group_id":"DG-A","split":"train","language":"en","text":"I cannot sign in to the portal","topic":"digital_service"},{"example_id":"F-003","group_id":"DG-B","split":"train","language":"ar","text":"الخدمة الإلكترونية سريعة وواضحة","topic":"digital_service"},{"example_id":"F-004","group_id":"DG-B","split":"train","language":"en","text":"The online service is clear and fast","topic":"digital_service"},{"example_id":"F-005","group_id":"PG-A","split":"train","language":"ar","text":"أحتاج معرفة حالة طلب التصريح","topic":"permit"},{"example_id":"F-006","group_id":"PG-A","split":"train","language":"en","text":"I need the status of my permit request","topic":"permit"},{"example_id":"F-007","group_id":"PG-B","split":"train","language":"ar","text":"تمت الموافقة على التصريح اليوم","topic":"permit"},{"example_id":"F-008","group_id":"PG-B","split":"train","language":"en","text":"The permit was approved today","topic":"permit"},{"example_id":"F-009","group_id":"HG-A","split":"train","language":"ar","text":"تأخر موعد العيادة هذا الصباح","topic":"health"},{"example_id":"F-010","group_id":"HG-A","split":"train","language":"en","text":"My clinic appointment was delayed","topic":"health"},{"example_id":"F-011","group_id":"HG-B","split":"train","language":"ar","text":"كانت خدمة العيادة ممتازة","topic":"health"},{"example_id":"F-012","group_id":"HG-B","split":"train","language":"en","text":"The clinic service was excellent","topic":"health"},{"example_id":"F-013","group_id":"TG-A","split":"train","language":"ar","text":"الحافلة لم تصل في الوقت المحدد","topic":"transport"},{"example_id":"F-014","group_id":"TG-A","split":"train","language":"en","text":"The bus did not arrive on time","topic":"transport"},{"example_id":"F-015","group_id":"TG-B","split":"train","language":"ar","text":"كانت الرحلة مريحة ومنظمة","topic":"transport"},{"example_id":"F-016","group_id":"TG-B","split":"train","language":"en","text":"The trip was comfortable and organised","topic":"transport"},{"example_id":"F-017","group_id":"DG-V","split":"validation","language":"ar","text":"لم يصل رمز التحقق الرقمي","topic":"digital_service"},{"example_id":"F-018","group_id":"PG-V","split":"validation","language":"en","text":"How can I renew the permit","topic":"permit"},{"example_id":"F-019","group_id":"HG-V","split":"validation","language":"ar","text":"أحتاج إعادة جدولة الموعد الصحي","topic":"health"},{"example_id":"F-020","group_id":"TG-V","split":"validation","language":"en","text":"The bus route has changed","topic":"transport"},{"example_id":"F-021","group_id":"DG-T","split":"test","language":"en","text":"The verification code did not arrive","topic":"digital_service"},{"example_id":"F-022","group_id":"PG-T","split":"test","language":"ar","text":"تأخر إصدار التصريح المطلوب","topic":"permit"},{"example_id":"F-023","group_id":"HG-T","split":"test","language":"en","text":"The clinic appointment was cancelled","topic":"health"},{"example_id":"F-024","group_id":"TG-T","split":"test","language":"ar","text":"توقفت الحافلة قبل المحطة","topic":"transport"}]

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        text = response.read().decode("utf-8")
    rows = list(csv.DictReader(io.StringIO(text)))
    DATA_SOURCE = "github_course_file"
except Exception as exc:
    rows = FALLBACK_ROWS
    DATA_SOURCE = f"embedded_fallback:{type(exc).__name__}"

print("Data source:", DATA_SOURCE)
print("Rows:", len(rows))
print("Topics:", Counter(row["topic"] for row in rows))
assert len(rows) >= 24
assert {"ar", "en"} <= {row["language"] for row in rows}

In [ ]:
def validate_splits(rows):
    required = {"train", "validation", "test"}
    group_owner = {}
    labels_by_split = {name: set() for name in required}
    counts = Counter()
    for row in rows:
        split, group, label = row["split"], row["group_id"], row["topic"]
        if split not in required:
            raise ValueError(f"Unknown split: {split}")
        previous = group_owner.setdefault(group, split)
        if previous != split:
            raise ValueError(f"Group leakage: {group}")
        labels_by_split[split].add(label)
        counts[split] += 1
    all_labels = set().union(*labels_by_split.values())
    for split in required:
        if labels_by_split[split] != all_labels:
            raise ValueError(f"Missing label in {split}")
    return {
        "rows": dict(counts),
        "groups": len(group_owner),
        "labels": sorted(all_labels),
        "group_overlap": 0,
    }

split_report = validate_splits(rows)
print(json.dumps(split_report, ensure_ascii=False, indent=2))
assert split_report["group_overlap"] == 0
print("Split contract=PASS")

**المتوقع:** كل split يحتوي الفئات الأربع، و`group_overlap` يساوي صفرًا.

## 2) TF-IDF Baseline

نضبط الـbaseline على Train ونقرأ Validation. لا نستخدم Test حتى خلية القياس النهائي بعد تثبيت المسار.

In [ ]:
train_rows = [row for row in rows if row["split"] == "train"]
validation_rows = [row for row in rows if row["split"] == "validation"]
test_rows = [row for row in rows if row["split"] == "test"]
LABELS = sorted({row["topic"] for row in rows})
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
baseline.fit(
    [row["text"] for row in train_rows],
    [row["topic"] for row in train_rows],
)
baseline_val_pred = baseline.predict([row["text"] for row in validation_rows])
baseline_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    baseline_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
print("Baseline validation macro-F1 (MEASURED_SMOKE):", round(baseline_val_f1, 4))
assert 0.0 <= baseline_val_f1 <= 1.0
print("Baseline=PASS")

## 3) تنزيل الـcheckpoint وتجهيز النموذج

النموذج عام ومجاني، ولا يحتاج Hugging Face token. عند ظهور warning بأن classification head جديدة فهذا متوقع: المشفر مدرب مسبقًا، أما رأس فئات بيان فيبدأ عشوائيًا.

- GPU: Full fine-tuning قصير.
- CPU: تجميد المشفر وتدريب الرأس فقط.

كلاهما تدريب فعلي، لكن نوعه يسجل في النتائج.

In [ ]:
MODEL_ID = "distilbert/distilbert-base-multilingual-cased"
MAX_LENGTH = 64
BATCH_SIZE = 4
MAX_TRAIN_STEPS = 6

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    )
except Exception as exc:
    raise RuntimeError(
        "Checkpoint download failed. Reconnect the runtime and run this cell once. "
        "No API key is required."
    ) from exc

TRAINING_MODE = "full_finetune" if DEVICE.type == "cuda" else "frozen_encoder_cpu"
if TRAINING_MODE == "frozen_encoder_cpu":
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False

model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Training mode:", TRAINING_MODE)
print(f"Trainable parameters: {trainable:,} / {total:,}")
assert trainable > 0

In [ ]:
def iter_batches(examples, batch_size, *, shuffle=False):
    indexes = np.arange(len(examples))
    if shuffle:
        np.random.default_rng(SEED).shuffle(indexes)
    for start in range(0, len(indexes), batch_size):
        yield [examples[index] for index in indexes[start:start + batch_size]]


def encode_batch(batch):
    encoded = tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(
        [label2id[row["topic"]] for row in batch], dtype=torch.long
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def predict(examples):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            encoded = encode_batch(batch)
            logits = model(**encoded).logits
            predictions.extend(logits.argmax(-1).cpu().tolist())
    return [id2label[index] for index in predictions]

print("Batch functions=PASS")

## 4) خطوة Fine-tuning فعلية

القيمة ليست في الوصول إلى رقم مرتفع من 24–40 مثالًا؛ القيمة في مرور البيانات والـlabels والـoptimizer والـbackpropagation بصورة صحيحة.

In [ ]:
learning_rate = 2e-5 if TRAINING_MODE == "full_finetune" else 5e-4
optimizer = AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=learning_rate,
)
model.train()
losses = []
train_steps = 0
for batch in iter_batches(train_rows, BATCH_SIZE, shuffle=True):
    optimizer.zero_grad(set_to_none=True)
    encoded = encode_batch(batch)
    output = model(**encoded)
    loss = output.loss
    if not torch.isfinite(loss):
        raise RuntimeError("Non-finite training loss")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], max_norm=1.0
    )
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
    train_steps += 1
    print(f"step={train_steps} loss={losses[-1]:.4f}")
    if train_steps >= MAX_TRAIN_STEPS:
        break

assert train_steps >= 1 and all(math.isfinite(value) for value in losses)
print("Transformer optimizer steps=PASS")

In [ ]:
transformer_val_pred = predict(validation_rows)
transformer_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    transformer_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
print("Transformer validation macro-F1 (MEASURED_SMOKE):", round(transformer_val_f1, 4))
print("Validation predictions:", list(zip(
    [row["topic"] for row in validation_rows], transformer_val_pred
)))
assert 0.0 <= transformer_val_f1 <= 1.0

## 5) القياس النهائي المصغر

بعد تثبيت الإعدادات السابقة نقيس Test مرة واحدة. لا نغير الإعدادات بناء على هذه النتيجة ثم نعيد تسميتها Frozen Test.

In [ ]:
test_truth = [row["topic"] for row in test_rows]
baseline_test_pred = baseline.predict([row["text"] for row in test_rows]).tolist()
transformer_test_pred = predict(test_rows)

results = {
    "result_type": "MEASURED_SMOKE",
    "data_source": DATA_SOURCE,
    "model_id": MODEL_ID,
    "device": str(DEVICE),
    "training_mode": TRAINING_MODE,
    "seed": SEED,
    "train_steps": train_steps,
    "mean_train_loss": float(np.mean(losses)),
    "baseline_validation_macro_f1": float(baseline_val_f1),
    "transformer_validation_macro_f1": float(transformer_val_f1),
    "baseline_test_macro_f1": float(f1_score(
        test_truth, baseline_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_macro_f1": float(f1_score(
        test_truth, transformer_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_accuracy": float(accuracy_score(test_truth, transformer_test_pred)),
    "limitations": [
        "synthetic tiny dataset",
        "short training smoke",
        "not an estimate of production quality",
    ],
}
print(json.dumps(results, ensure_ascii=False, indent=2))
Path("day2_classification_metrics.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

لا تستنتج أن Transformer «فشل» أو «نجح إنتاجيًا» من هذه العينة. سجل الفرق، ثم اكتب ما تحتاجه لتقدير موثوق: بيانات أكثر، عدة بذور، slices، وتقييم مجمد.

## مستويات التحدي

- **Core:** الخلايا السابقة فقط.
- **Explore:** على GPU ارفع `MAX_TRAIN_STEPS` بعد حفظ نتيجة Core وقارن الزمن والـvalidation.
- **Distinction:** نفذ ثلاث بذور وسجل mean ± range؛ لا تستخدم Test لاختيار البذرة.

In [ ]:
core_checks = {
    "split_isolation": split_report["group_overlap"] == 0,
    "baseline_valid": 0.0 <= baseline_val_f1 <= 1.0,
    "training_ran": train_steps >= 1,
    "loss_is_finite": all(math.isfinite(value) for value in losses),
    "validation_valid": 0.0 <= transformer_val_f1 <= 1.0,
    "result_is_honest": results["result_type"] == "MEASURED_SMOKE",
    "bilingual_data": {"ar", "en"} <= {row["language"] for row in rows},
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY2_NOTEBOOK3_CORE=PASS")

## نقطة GitHub

1. احفظ نسخة notebook في مستودعك بالاسم نفسه.
2. لا ترفع Hugging Face cache أو model weights.
3. ارفع `day2_classification_metrics.json` ضمن `reports/` إذا لم يحتو بيانات شخصية.
4. حدّث `DECISIONS.md` بنوع التدريب والسبب وحدود العينة.
5. لا تعمل commit النهائي لليوم حتى تكمل دفتر NER وQA.

**التالي:** [مختبر NER وQA](04_ner_and_qa.ipynb).